In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error ,mean_absolute_error ,r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,SimpleRNN,Dropout,Dense

In [ ]:
data = pd.read_csv('Google_Stock_Price_Train.csv',thousands=",")
data.head()

In [ ]:
data.info()

In [ ]:
data['Date']= pd.to_datetime(data['Date'])

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data.plot(x='Date',y=['Open','High','Low','Close'],figsize=(12,6))
plt.grid(True)
plt.show()

In [ ]:
data.drop('Date',axis=1,inplace=True)

In [ ]:
data.info()

In [ ]:
# normalize the data using MinMaxScaler
sc =MinMaxScaler()
data=sc.fit_transform(data)

In [ ]:
# Check for missing values
def create_seq (data,idx,seqlen=60): # 60 days of historical data to predict the next day's price
    # seqlen is the number of previous time steps to consider for predicting the next value
    x,y=[],[]
    for i in range(seqlen,len(data)):
        x.append(data[i-seqlen:i])
        y.append(data[i,idx])
    return np.array(x),np.array(y)
x,y= create_seq(data,3)
x

In [ ]:
y

In [ ]:
split = int(0.8*len(data)) # 80% for training and 20% for testing
x_train = x[:split]
x_test = x[split:]
y_train = y[:split]
y_test = y[split:]

In [ ]:
# using SimpleRNN
# SimpleRNN is a type of recurrent neural network that is used for processing sequential data. 
# It has a simple architecture and is suitable for tasks where the relationships between time steps are not too complex. 
# In this model, we will use multiple layers of SimpleRNN to capture the temporal dependencies in the stock price data, 
# along with Dropout layers to prevent overfitting. 
# Finally, we will add a Dense layer to output the predicted stock price.

model_sr = Sequential()
model_sr.add(SimpleRNN(64,return_sequences=True,activation='tanh',input_shape=(x_train.shape[1],x_train.shape[2])))
model_sr.add(Dropout(0.2))
model_sr.add(SimpleRNN(64,return_sequences=True,activation='tanh'))
model_sr.add(Dropout(0.2))
model_sr.add(SimpleRNN(64,return_sequences=True,activation='tanh'))
model_sr.add(Dropout(0.2))
model_sr.add(SimpleRNN(64))
model_sr.add(Dropout(0.2))
model_sr.add(Dense(1))
model_sr.compile(optimizer='adam',loss='mse')
model_sr.summary()

In [ ]:
# h_sr stands for history of the SimpleRNN model training process. 
# It contains information about the training and validation loss and metrics for each epoch.
h_sr= model_sr.fit(
    x_train,y_train,
    batch_size=32,
    epochs=50,
    validation_data=(x_test,y_test),
    verbose=1
)

In [ ]:
y_pred= model_sr.predict(x_test)

In [ ]:
close_sc = MinMaxScaler()
close_sc.min_,close_sc.scale_ = sc.min_[3:4],sc.scale_[3:4]

In [ ]:
# Inverse transform the predicted and actual values to get the original scale of stock prices
predicted = close_sc.inverse_transform(y_pred)
actual = close_sc.inverse_transform(y_test.reshape(-1,1))

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(actual,label="Actual")
plt.plot(predicted,label="Predicted")
plt.grid(True)
plt.show()

In [ ]:
print("RMSE : ",np.sqrt(mean_squared_error(actual,predicted)))
print("R2 score ",r2_score(actual,predicted))
print("MAE : ",mean_absolute_error(actual,predicted))

In [ ]:
# using LSTM
# LSTM (Long Short-Term Memory) is a type of recurrent neural network that is designed to capture long-term dependencies in sequential data. 
# It has a more complex architecture than SimpleRNN, with memory cells and gating mechanisms that allow it to retain information over longer time periods. 
# In this model, we will use multiple layers of LSTM to capture the temporal dependencies in the stock price data, along with Dropout layers to prevent overfitting. 
# Finally, we will add a Dense layer to output the predicted stock price.
model_lstm = Sequential()
model_lstm.add(LSTM(64,return_sequences=True,input_shape=(x_train.shape[1],x_train.shape[2])))
model_lstm.add(Dropout(0.2))
model_lstm.add(LSTM(64))
model_lstm.add(Dropout(0.1))
model_lstm.add(Dense(1))
model_lstm.compile(optimizer='adam',loss='mse')
model_lstm.summary()

In [ ]:
# h_lstm stands for history of the LSTM model training process.
h_lstm= model_lstm.fit(
    x_train,y_train,
    batch_size=32,
    epochs=50,
    validation_data=(x_test,y_test),
    verbose=1
)

In [ ]:
y_pred_lstm= model_lstm.predict(x_test)

In [ ]:
# Inverse transform the predicted and actual values to get the original scale of stock prices
predicted_lstm = close_sc.inverse_transform(y_pred_lstm)
actual_lstm = close_sc.inverse_transform(y_test.reshape(-1,1))

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(actual_lstm,label="Actual")
plt.plot(predicted_lstm,label="Predicted")
plt.grid(True)
plt.show()

In [ ]:
# Evaluate the performance of the LSTM model using RMSE, R2 score, and MAE
print("RMSE : ",np.sqrt(mean_squared_error(actual_lstm,predicted_lstm)))
print("R2 score ",r2_score(actual_lstm,predicted_lstm))
print("MAE : ",mean_absolute_error(actual_lstm,predicted_lstm))